# Validate mobility with a distance law and flow models

**Question.** From raw GPS tracks, how do I check that the movement of my sample follows a distance law, with an honest uncertainty, and that a gravity or radiation model reproduces the flows between places?

**You will get** the full cycle on one dataset: cleaned tracks, stops, the distribution of trip lengths fitted with confidence intervals and compared with alternative forms, flows between places, a gravity fit with its distance-decay parameter and confidence interval, and a held-out comparison of the gravity and the radiation model.

**Prerequisites.** [Clean noisy GPS data and find stops](clean-and-detect-stops.md) and [How the fits work](fitting-methods.md).

**Data.** [GeoLife](https://www.microsoft.com/en-us/download/details.aspx?id=52367), GPS tracks of Beijing residents collected by Microsoft Research Asia. `fastmob.data.load_geolife` reads the first 60 trips of users 000 to 029 straight from the public archive (about 2.7 million position fixes; the first run downloads several tens of megabytes and takes a few minutes). Check the dataset page for its terms of use before redistributing the data. If the download fails the notebook stops; it never substitutes other data.

In [1]:
%pip install -q 'fastmob>=0.3.4' pandas

Note: you may need to restart the kernel to use updated packages.


## Load and clean

We keep fixes inside the urban area of Beijing and drop the ones that would need an impossible speed (above 200 km/h, as in the cleaning notebook).

In [2]:
import fastmob
import numpy as np
import pandas as pd
from fastmob.core import FlowDataFrame, Locations
from fastmob.data import load_geolife
from fastmob.measures.evaluation import compare_flow_models
from fastmob.measures.fitting import (
    compare_distance_distributions,
    fit_distance_distribution,
    likelihood_ratio_test,
)
from fastmob.models import Gravity, Radiation

users = [f"{i:03d}" for i in range(30)]
fixes = load_geolife(users=users, max_trips_per_user=60).to_pandas()
clean = fastmob.filter(fixes, max_speed_kmh=200)
clean = clean[clean["lat"].between(39.6, 40.3) & clean["lng"].between(116.0, 116.8)]
print(f"{len(fixes):,} fixes, {len(clean):,} kept, {clean['uid'].nunique()} users")

2,728,148 fixes, 2,012,465 kept, 30 users


## From fixes to trips

At one fix every few seconds, consecutive fixes are not trips. A trip goes from one stop to the next, so we find where each person stayed at least 20 minutes within 200 m (`stay_locations`) and treat each pair of consecutive stops of a person as one trip.

In [3]:
stays = fastmob.stay_locations(clean, minutes_for_a_stop=20, spatial_radius_km=0.2)
print(f"{len(stays):,} stops by {stays['uid'].nunique()} people, {len(stays) - stays['uid'].nunique():,} trips between consecutive stops")

3,367 stops by 30 people, 3,337 trips between consecutive stops


## Step 1: the distribution of trip lengths

`jump_lengths` on the stops gives the straight-line length of each trip. We fit the five families by maximum likelihood from 0.2 km upwards (the radius of a stop: two stops closer than that are the same place, so shorter trips only measure the GPS noise) and rank them by AIC. A family with a parameter on the edge of its search range is a degenerate fit: it is listed last, with no Akaike weight.

In [4]:
trip_lengths = fastmob.jump_lengths(stays)
lengths = np.concatenate(trip_lengths["jump_lengths"].to_list())
owner = np.repeat(trip_lengths["uid"].to_numpy(), trip_lengths["jump_lengths"].map(len).to_numpy())
usable = np.isfinite(lengths) & (lengths > 0)
lengths, owner = lengths[usable], owner[usable]
print(f"{len(lengths):,} trips, median {np.median(lengths):.1f} km, longest {lengths.max():.0f} km")
compare_distance_distributions(lengths, xmin=0.2).to_pandas().round(2)

3,335 trips, median 1.3 km, longest 62 km


,family,n_parameters,log_likelihood,aic,bic,delta_aic,akaike_weight,ks_statistic,gof_p_value,at_bound,degenerate
0,truncated_powerlaw,3,-14354.53,28715.06,28732.84,0.00,0.97,0.05,None,,False
1,lognormal,2,-14359.07,28722.13,28733.98,7.07,0.03,0.05,None,,False
2,shifted_powerlaw,2,-14364.90,28733.79,28745.64,18.73,0.00,0.05,None,,False
3,powerlaw,1,-14572.66,29147.33,29153.25,432.26,0.00,0.10,None,,False
4,exponential,1,-14846.16,29694.33,29700.25,979.26,0.00,0.23,None,,False


The 3,335 trips have a median of 1.3 km and the longest is 62 km; 2,765 of them are at least 0.2 km and enter the fit. The truncated power law ranks first, with an Akaike weight of 0.97, the log-normal is 7 AIC units behind, the shifted power law 19, and the plain power law and the exponential are far behind (432 and 979). No family is degenerate (`at_bound` is empty, `degenerate` is False), so the ranking can be read. The gap between the first two is small for 2,765 trips and the values of one person are not independent, so the data do not decide firmly between the truncated power law and the log-normal; the others are clearly worse.

In [5]:
fit = fit_distance_distribution(lengths, "truncated_powerlaw", xmin=0.2, n_bootstrap=100, groups=owner)
shifted = fit_distance_distribution(lengths, "shifted_powerlaw", xmin=0.2, n_bootstrap=0)
test = likelihood_ratio_test(shifted, fit_distance_distribution(lengths, "truncated_powerlaw", xmin=0.2, n_bootstrap=0))
print(f"{fit.n:,} trips between {fit.xmin:.3f} and {fit.xmax:.0f} km; KS distance {fit.ks_statistic:.3f}; at the edge: {fit.at_bound or 'none'}")
print(f"cutoff against no cutoff: statistic {test.statistic:.1f}, p = {test.p_value:.2g}")
print(fit.profile("kappa"))
fit.summary().to_pandas().round(3)

2,765 trips between 0.200 and 62 km; KS distance 0.051; at the edge: none
cutoff against no cutoff: statistic 20.7, p = 2.6e-06


kappa in [22.52, 52.82] (95%), estimate 31.13


,parameter,estimate,std_error,ci_low,ci_high,at_bound
0,r0,0.748,0.464,0.005,1.615,False
1,beta,1.306,0.334,0.673,1.972,False
2,kappa,31.130,130.876,11.982,616.056,False


The fit has no parameter on the edge of its range. The cutoff is `kappa` = 31 km, and the likelihood-ratio test of a cutoff against none gives a statistic of 20.7 (p = 3e-6), so unlike the Brightkite jump lengths these trips do bend at a distance. The interval of `kappa` depends on what is resampled. The profile likelihood, which treats the trips as independent, gives 23 to 53 km. The bootstrap that resamples people gives 12 to 616 km (standard error 131 km): the 30 people have different habits, so the cutoff is far less certain than the independent interval suggests. Report the interval that resamples people. The exponent `beta` = 1.31 (0.67 to 1.97) and the shift `r0` = 0.75 km (0.005 to 1.6) are also wide. The KS distance of 0.051 is the largest gap between the fitted and the empirical cumulative distribution, 5 percentage points.

## Step 2: flows between places

To test the mobility itself, we count trips between places. We use H3 cells of resolution 7 (about 1.2 km on a side) and keep the 40 cells with most stops, whose number of stops is their relevance (the size of the place). `trajectory_to_od` on the stops pairs consecutive stops of a person, so one row of the flow table is the number of trips from one cell to another. A `Locations` catalogue holds the cell centres and relevance, which is what the gravity model needs.

In [6]:
def busiest_cells(frame, n=40):
    """The n cells with most stops, with their number of stops as relevance."""
    counted = fastmob.latlng_to_h3(frame, resolution=7).groupby("h3_cell").size()
    return counted.rename("relevance").nlargest(n).rename_axis("tile_id")


def cell_flows(frame, cells):
    """Trips between the given cells, for the given stops, with their catalogue."""
    trips = fastmob.trajectory_to_od(frame, resolution=7).rename(columns={"count": "flow"})
    trips = trips[
        trips["origin"].isin(cells.index)
        & trips["destination"].isin(cells.index)
        & (trips["origin"] != trips["destination"])
    ].reset_index(drop=True)
    centres = fastmob.h3_to_latlng(pd.DataFrame({"h3_cell": cells.index})).rename(
        columns={"h3_cell": "tile_id", "center_lat": "lat", "center_lng": "lng"}
    )
    table = (
        centres.join(cells, on="tile_id")
        .join(trips.groupby("origin")["flow"].sum().rename("tot_outflow"), on="tile_id")
        .fillna({"tot_outflow": 0})
    )
    catalogue = Locations.from_tessellation(table, tile_id_col="tile_id", lat_col="lat", lng_col="lng")
    return FlowDataFrame(trips, locations=catalogue, tile_id="tile_id"), catalogue


cells = busiest_cells(stays)
observed, catalogue = cell_flows(stays, cells)
table = observed.to_pandas()
print(f"{len(table)} pairs of cells with trips out of {40 * 39}; {int(table['flow'].sum())} trips; largest flow {int(table['flow'].max())}")

331 pairs of cells with trips out of 1560; 1570 trips; largest flow 176


A singly constrained gravity model matches the number of trips leaving each cell and fits how they spread among destinations, `flow ∝ relevance(destination)^a * distance^gamma`. `Gravity.fit` adds the pairs of cells without trips as zeros, gives robust standard errors and a confidence interval for `gamma`, the distance-decay parameter.

In [7]:
model = Gravity(gravity_type="singly constrained")
model.fit(observed)
print(model.fit_statistics())
model.fit_summary().to_pandas().round(3)

{'n_observations': 1560, 'log_likelihood': -1971.2258359236778, 'deviance': 3054.841933895, 'pseudo_r_squared': 0.6374740635954954, 'dispersion': 5.189883995618827, 'robust_errors': True}


,parameter,estimate,std_error,ci_low,ci_high,p_value
0,destination_exp,0.746,0.085,0.580,0.911,0.0
1,distance_coefficient,-1.378,0.148,-1.667,-1.089,0.0


Of the 1,560 pairs of the 40 cells, 331 have trips; the other 1,229 enter the fit as zeros. The distance coefficient is -1.38 with a 95% interval of -1.67 to -1.09: the flow between two cells falls as roughly the 1.4th power of their distance, and an exponent of 1 or of 2 would both be hard to defend at this interval. The destination exponent is 0.75 (0.58 to 0.91). The dispersion is 5.2, well above 1, so the Poisson standard errors would be too small and the robust ones, which are reported, are the ones to use. The pseudo R-squared of 0.64 is mostly the size of the origins and does not show that the distance law holds; the interval of the coefficient does. The relevance is the number of stops at the destination, which are made by the same people as the trips, so the destination exponent is partly circular; the distance coefficient is much less affected. The exponent describes the distances between the centres of 1.2 km cells.

## Step 3: does the model predict flows it has not seen?

The check above fits and describes the same trips. For a test of the model, we split the people in two halves. The cells and their relevance are chosen from the first half, the gravity model is fitted on the first half's trips, and both models are scored on the second half's trips. The radiation model has no parameter to fit and uses the same catalogue, so it is the baseline. `compare_flow_models` scores every model on all the pairs of the cells, so the rows can be compared.

In [8]:
people = np.sort(stays["uid"].unique())
in_first = stays["uid"].isin(people[::2])
first, second = stays[in_first], stays[~in_first]

cells_first = busiest_cells(first)
train_flows, train_catalogue = cell_flows(first, cells_first)
test_flows, test_catalogue = cell_flows(second, cells_first)

fitted = Gravity(gravity_type="singly constrained")
fitted.fit(train_flows)
print(f"trips used to fit: {int(train_flows.to_pandas()['flow'].sum())}; trips held out: {int(test_flows.to_pandas()['flow'].sum())}")
print(fitted.fit_summary().to_pandas().round(3).to_string(index=False))

models = {
    "gravity (fitted on the first half)": fitted.generate(test_catalogue, out_format="flows"),
    "radiation": Radiation().generate(test_catalogue, out_format="flows"),
}
compare_flow_models(test_flows, models).to_pandas().set_index("model").round(3)

trips used to fit: 963; trips held out: 381
           parameter  estimate  std_error  ci_low  ci_high  p_value
     destination_exp     0.708      0.077   0.557    0.860      0.0
distance_coefficient    -1.327      0.181  -1.682   -0.971      0.0


,n_pairs,observed_flow,model_flow,cpc,cpl,r_squared,spearman,poisson_deviance,pseudo_r_squared
model,,,,,,,,,
gravity (fitted on the first half),1560,381.0,381.0,0.411,0.198,0.390,0.381,810.207,0.725
radiation,1560,381.0,346.0,0.446,0.443,0.342,0.428,5730.897,-0.949


The gravity model fitted on the first half's 963 trips gives a distance coefficient of -1.33 (-1.68 to -0.97), inside the interval of the fit on everyone, and a destination exponent of 0.71. Both models are given the number of trips that leave each cell in the second half (381 trips), so the comparison is about how those trips spread over destinations. The scores disagree. The gravity model is far better by the Poisson deviance (810 against 5,731; the pseudo R-squared is 0.73 against -0.95, which means the radiation flows are worse than a constant) and by the R-squared (0.39 against 0.34). The radiation model is slightly better by the common part of commuters (0.45 against 0.41) and the rank correlation (0.43 against 0.38). Do not read the CPL here: both models give expected flows, which are positive for every pair, so their CPL (0.20 and 0.44) reflects the number of pairs and says nothing about the model. With 381 held-out trips the scores have a large sampling error, so the honest summary is that the fitted gravity model predicts the flows at least as well as the parameter-free radiation model, and clearly better in likelihood, and that the data are too few to say more.

## Interpretation and limits

- A distance law is a statement about a range. The trip lengths are fitted from 0.2 km to 60 km, and the family that wins on this range need not win on another; say which range you used.
- The intervals of the distance fit resample people, because the trips of one person are not independent. With 30 people they are wide, and they describe these people, not Beijing.
- GeoLife is a sample of a research project's volunteers, mostly around one university; the flows are a sample of their movement, not of the city, and the number of stops at a cell is partly a measure of where these people live and work. The relevance is not independent of the flows, so the destination exponent partly reflects circularity (see How the fits work).
- The hold-out has a few hundred trips, so the scores have a large sampling error that the table does not show. A difference between two models must be large and consistent across scores before you believe it.
- Compare the distance exponent with the literature only for flows of the same kind, zone size and range.

## What to report

The data and the filtering, the definition of a stop and a trip, the number of trips and people, the range and family of the distance distribution with its interval (and that people were resampled), the zones and their relevance, the deterrence function and constraint of the gravity model, whether zero pairs were included, the distance coefficient with its interval, the dispersion, the hold-out design and the scores of the baseline next to those of the model. The checklist in [How the fits work](fitting-methods.md) is the complete list.

## Next steps

- How the estimators work and what to report: [How the fits work](fitting-methods.md).
- Check flows against a reference table: [Validate against census flows](validate-against-census-flows.md).
- Compare the fitted law with other data: [Compare to mobility laws](compare-to-mobility-laws.md).

## Related API

[`stay_locations`](../reference/preprocessing.md#fastmob.preprocessing.stay_locations) ·
[`jump_lengths`](../reference/measures/individual.md#fastmob.measures.individual.jump_lengths) ·
[`fit_distance_distribution`](../reference/fitting.md#fastmob.measures.fitting.distance_distribution.fit_distance_distribution) ·
[`compare_distance_distributions`](../reference/fitting.md#fastmob.measures.fitting.distance_distribution.compare_distance_distributions) ·
[`likelihood_ratio_test`](../reference/fitting.md#fastmob.measures.fitting.distribution_tests.likelihood_ratio_test) ·
[`Gravity.fit`](../reference/models.md#fastmob.models.gravity.Gravity.fit) ·
[`compare_flow_models`](../reference/measures/evaluation.md#fastmob.measures.evaluation.compare_flow_models)